# visual coding neuropixel sessions check

In [1]:
import os
import pandas as pd

from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache

data_directory = os.path.expanduser("~/allen_neuropixels_data")
manifest_path = os.path.join(data_directory, "manifest.json")

cache = EcephysProjectCache.from_warehouse(manifest=manifest_path)

sessions = cache.get_session_table()
channels = cache.get_channels()

print(f"Sessions: {len(sessions)}")
print(f"Channels: {len(channels)}")

channels.head()

/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sessions: 58
Channels: 123224


,ecephys_probe_id,local_index,probe_horizontal_position,probe_vertical_position,anterior_posterior_ccf_coordinate,dorsal_ventral_ccf_coordinate,left_right_ccf_coordinate,ecephys_structure_id,ecephys_structure_acronym,ecephys_session_id,lfp_sampling_rate,phase,sampling_rate,has_lfp_data,unit_count
id,,,,,,,,,,,,,,,
849705558,792645504,1,11,20,8165.0,3314.0,6862.0,215.0,APN,779839471,1250.001479,3a,30000.035489,True,0
849705560,792645504,2,59,40,8162.0,3307.0,6866.0,215.0,APN,779839471,1250.001479,3a,30000.035489,True,0
849705562,792645504,3,27,40,8160.0,3301.0,6871.0,215.0,APN,779839471,1250.001479,3a,30000.035489,True,0
849705564,792645504,4,43,60,8157.0,3295.0,6875.0,215.0,APN,779839471,1250.001479,3a,30000.035489,True,0
849705566,792645504,5,11,60,8155.0,3288.0,6879.0,215.0,APN,779839471,1250.001479,3a,30000.035489,True,0


In [7]:
coverage = (
    channels.groupby(["ecephys_session_id", "ecephys_structure_acronym"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=sessions.index, fill_value=0)
)

coverage["n_regions"] = (coverage > 0).sum(axis=1)

coverage = coverage.sort_values("n_regions", ascending=False)

coverage.head(10)

ecephys_structure_acronym,APN,BMAa,CA1,CA2,CA3,COAa,COApm,CP,DG,Eth,...,VISmmp,VISp,VISpm,VISrl,VL,VPL,VPM,ZI,grey,n_regions
id,,,,,,,,,,,,,,,,,,,,,
794812542,47,0,287,5,67,0,0,0,214,0,...,0,96,89,100,0,0,0,0,0,24
787025148,145,0,315,0,80,0,0,0,219,0,...,0,98,0,77,0,0,0,0,10,24
781842082,146,0,356,9,109,0,0,0,249,0,...,0,85,0,93,0,0,0,0,12,24
757216464,161,0,387,0,87,0,0,0,238,30,...,0,95,0,86,0,0,0,0,0,24
768515987,150,0,359,9,97,0,0,0,210,30,...,0,90,0,93,0,0,0,0,0,23
719161530,211,0,364,9,102,0,0,0,160,1,...,0,85,86,75,14,0,24,0,36,23
778240327,50,0,489,5,77,0,0,0,201,0,...,0,95,91,86,0,0,0,4,1,23
771160300,11,0,460,12,84,0,0,0,113,0,...,0,94,91,0,0,0,0,12,0,23
755434585,65,0,311,0,98,0,0,0,207,0,...,0,94,86,56,0,0,0,0,140,22


### export table

In [9]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(32, 20))
ax.axis("off")

table = ax.table(
    cellText=coverage.values,
    colLabels=coverage.columns,
    rowLabels=coverage.index,
    loc="center",
    cellLoc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(5)
table.scale(1, 1.3)

plt.savefig(
    "visual_coding_coverage.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.2
)

plt.close()

coverage.to_csv("visual_coding_coverage.csv", index=True)